# HyDE — Hypothetical Document Embeddings (Native LangChain & LCEL)

This notebook demonstrates the complete **HyDE (Hypothetical Document Embeddings)** pipeline built **entirely with native LangChain components and LCEL**, without requiring any custom `hyde.py` wrapper class.

```
User Query
    │
    ▼
LLM generates a Hypothetical Document (authoritative answer text)
    │
    ▼
Hypothetical Document is embedded and queries ChromaDB (via retriever)
    │
    ▼
Retrieved real documents are formatted into context
    │
    ▼
Generation LLM produces the final grounded answer
```

**Why Native LCEL for HyDE?**  
In LangChain, a retriever is a `Runnable` that takes a string input and outputs a list of `Document`s. Since the hypothetical document generator (`prompt | llm | StrOutputParser()`) outputs a string, we can chain them directly:
```python
hyde_retriever = hyde_prompt | document_llm | StrOutputParser() | base_retriever
```
No custom classes, no helper files, and no complex wrappers needed!

## 1. Imports & Environment Setup

In [1]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

# Load API keys from .env file
load_dotenv()

True

## 2. Knowledge Base Documents

A curated set of 17 documents across AI, Physics, Tech, and Medicine.

In [2]:
documents = [
    # --- AI ---
    Document(
        page_content=(
            "The Transformer architecture, introduced in 'Attention Is All You Need' (Vaswani et al., 2017), "
            "replaces recurrent layers entirely with self-attention mechanisms. Each token attends to every "
            "other token in the sequence simultaneously, computing weighted sums based on query-key dot products. "
            "This parallelism enables efficient training on long sequences and has become the backbone of modern LLMs."
        ),
        metadata={"source": "AI", "topic": "Transformers"}
    ),
    Document(
        page_content=(
            "Multi-head attention allows the model to jointly attend to information from different representation "
            "subspaces at different positions. Each head learns distinct attention patterns — some heads track "
            "syntactic dependencies while others capture semantic relationships. The outputs of all heads are "
            "concatenated and linearly projected to produce the final representation."
        ),
        metadata={"source": "AI", "topic": "Attention Mechanisms"}
    ),
    Document(
        page_content=(
            "Reinforcement Learning from Human Feedback (RLHF) is a training paradigm where a reward model "
            "is first trained on human preference data, then used to fine-tune a language model via PPO. "
            "This process aligns model outputs with human values and has been central to making LLMs like "
            "ChatGPT safe and helpful in practice."
        ),
        metadata={"source": "AI", "topic": "RLHF"}
    ),
    Document(
        page_content=(
            "Retrieval-Augmented Generation (RAG) combines a retrieval system with a generative model. "
            "Given a query, relevant documents are fetched from an external knowledge base and prepended "
            "to the prompt as context. This grounds the model's response in factual information and reduces "
            "hallucination without requiring costly retraining."
        ),
        metadata={"source": "AI", "topic": "RAG"}
    ),
    Document(
        page_content=(
            "Large Language Models (LLMs) are trained on massive text corpora using next-token prediction. "
            "Scaling laws show that model performance improves predictably with increases in parameters, "
            "data, and compute. Emergent capabilities — such as few-shot reasoning and chain-of-thought — "
            "appear at certain scale thresholds and were not explicitly trained for."
        ),
        metadata={"source": "AI", "topic": "LLMs"}
    ),

    # --- Physics ---
    Document(
        page_content=(
            "Quantum entanglement is a phenomenon where two or more particles become correlated such that "
            "the quantum state of one cannot be described independently of the others, even when separated "
            "by large distances. Measuring one particle's state instantaneously determines the correlated "
            "property of the other, a fact Einstein famously called 'spooky action at a distance'."
        ),
        metadata={"source": "Physics", "topic": "Quantum Entanglement"}
    ),
    Document(
        page_content=(
            "A black hole is a region of spacetime where gravity is so strong that nothing — not even light — "
            "can escape once it crosses the event horizon. Stellar black holes form when massive stars collapse "
            "under their own gravity. Supermassive black holes, found at the centres of most galaxies, "
            "can contain billions of solar masses and are thought to power quasars."
        ),
        metadata={"source": "Physics", "topic": "Black Holes"}
    ),
    Document(
        page_content=(
            "The second law of thermodynamics states that the total entropy of an isolated system never decreases "
            "over time. Entropy can be understood as a measure of disorder or the number of microscopic "
            "configurations consistent with a macroscopic state. This law underpins the irreversibility of "
            "natural processes such as heat flow and chemical reactions."
        ),
        metadata={"source": "Physics", "topic": "Thermodynamics"}
    ),
    Document(
        page_content=(
            "Einstein's special theory of relativity postulates that the laws of physics are identical in all "
            "inertial frames and that the speed of light in a vacuum is constant regardless of the observer's "
            "motion. Consequences include time dilation, length contraction, and the equivalence of mass and "
            "energy expressed as E = mc²."
        ),
        metadata={"source": "Physics", "topic": "Special Relativity"}
    ),

    # --- Tech ---
    Document(
        page_content=(
            "Microservices architecture structures an application as a collection of small, independently "
            "deployable services, each responsible for a specific business capability. Services communicate "
            "over lightweight APIs (typically HTTP/REST or gRPC). This approach improves fault isolation, "
            "enables technology heterogeneity, and allows teams to scale individual components independently."
        ),
        metadata={"source": "Tech", "topic": "Microservices"}
    ),
    Document(
        page_content=(
            "Kubernetes is an open-source container orchestration platform that automates the deployment, "
            "scaling, and management of containerised applications. It groups containers into Pods, manages "
            "their lifecycle via controllers, and exposes them through Services. The control plane maintains "
            "the desired state of the cluster using a reconciliation loop."
        ),
        metadata={"source": "Tech", "topic": "Kubernetes"}
    ),
    Document(
        page_content=(
            "LLM inference at scale requires specialised hardware and serving strategies. Tensor parallelism "
            "splits model weights across multiple GPUs to fit large models in memory. Continuous batching "
            "improves GPU utilisation by dynamically grouping requests. Quantisation (INT8/INT4) reduces "
            "memory footprint at the cost of minor precision loss, enabling deployment on fewer accelerators."
        ),
        metadata={"source": "Tech", "topic": "LLM Inference"}
    ),
    Document(
        page_content=(
            "Modern semiconductor fabrication uses extreme ultraviolet (EUV) lithography to etch features "
            "at the 3–5 nm process node. TSMC and Samsung lead production at these scales. Each wafer "
            "passes through hundreds of deposition, etching, and doping steps inside cleanrooms rated "
            "ISO Class 1. Yield improvement is critical: a single defect can render an entire die unusable."
        ),
        metadata={"source": "Tech", "topic": "Semiconductors"}
    ),

    # --- Medicine ---
    Document(
        page_content=(
            "CRISPR-Cas9 is a genome editing tool derived from a bacterial immune defence mechanism. "
            "A guide RNA directs the Cas9 nuclease to a specific DNA sequence, where it introduces a "
            "double-strand break. The cell's repair machinery then inserts, deletes, or replaces genetic "
            "material. Clinical trials are underway for sickle cell disease, certain cancers, and "
            "hereditary blindness."
        ),
        metadata={"source": "Medicine", "topic": "CRISPR"}
    ),
    Document(
        page_content=(
            "mRNA vaccines work by delivering synthetic messenger RNA into cells, instructing them to "
            "produce a target antigen (e.g., the SARS-CoV-2 spike protein). The immune system mounts a "
            "response against the antigen, creating memory cells. The mRNA itself degrades within days "
            "and does not integrate into the host genome, making this platform both effective and safe."
        ),
        metadata={"source": "Medicine", "topic": "mRNA Vaccines"}
    ),
    Document(
        page_content=(
            "Brain-computer interfaces (BCIs) record neural signals and translate them into control commands "
            "for external devices. Implanted electrode arrays such as Utah arrays can record from hundreds "
            "of neurons simultaneously. Clinical applications include restoring motor function in paralysed "
            "patients and treating refractory depression via deep brain stimulation."
        ),
        metadata={"source": "Medicine", "topic": "Neural Implants"}
    ),
    Document(
        page_content=(
            "Antibiotic resistance arises when bacteria evolve mechanisms to survive drug exposure — including "
            "enzymatic drug inactivation, efflux pumps, and target modification. Overuse and misuse of "
            "antibiotics accelerates resistance. The WHO classifies antimicrobial resistance as one of the "
            "greatest global health threats, with carbapenem-resistant Enterobacteriaceae among the most "
            "critical pathogens."
        ),
        metadata={"source": "Medicine", "topic": "Antibiotic Resistance"}
    )
]

print(f"Total documents loaded: {len(documents)}")

Total documents loaded: 17


## 3. ChromaDB Vector Store & Base Retriever

We embed our 17 documents using HuggingFace embeddings (`sentence-transformers/all-MiniLM-L6-v2`) and create a base retriever returning top-3 documents.

In [3]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="hyde_native_demo"
)

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("ChromaDB collection and base retriever ready.")

c:\Users\sujat\projects\AI-Main\Advanced_Rag_Codes\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 930.54it/s]


ChromaDB collection and base retriever ready.


## 4. LLM Initialization

- `document_llm`: Low temperature (0.0) to generate a factual, structured hypothetical document.
- `generation_llm`: Slightly higher temperature (0.5) to synthesize the final user-facing response.

In [4]:
if os.getenv("GROQ_API_KEY"):
    document_llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0.0)
    generation_llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0.5)
else:
    document_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    generation_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

print("LLMs initialized.")

LLMs initialized.


## 5. Native HyDE Retriever using LCEL (No `hyde.py`!)

In native LangChain, HyDE is composed using pure LCEL:
1. **`hyde_prompt`**: Instructs the LLM to write an authoritative passage answering the query.
2. **`hypothetical_doc_generator`**: `hyde_prompt | document_llm | StrOutputParser()` outputs the text.
3. **`hyde_retriever`**: Pipes that text directly into `base_retriever`.

In [5]:
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an expert document writer. Given a user query, your task is to generate a hypothetical document "
        "that would directly and thoroughly answer the query.\n"
        "The document should be written as if it were a real, authoritative passage retrieved from a knowledge base — "
        "not a response to the user, but a self-contained piece of text that contains the answer.\n"
        "Write in a factual, informative tone. Do not include phrases like 'Based on your query' or 'Here is a document'.\n"
        "Output only the hypothetical document text, nothing else."
    )),
    ("human", "query: {query}")
])

# Step A: Generate hypothetical document text string
hypothetical_doc_generator = hyde_prompt | document_llm | StrOutputParser()

# Step B: Pipe the generated document string directly into the base retriever!
hyde_retriever = hypothetical_doc_generator | base_retriever

print("Native HyDE retriever ready.")

Native HyDE retriever ready.


## 6. Inspecting the HyDE Process

Let's see the hypothetical document generated by the LLM, and examine the documents retrieved using it.

In [6]:
query = "How do transformer models use attention to process sequences?"

# 1. Preview the hypothetical document
hypo_doc = hypothetical_doc_generator.invoke({"query": query})
print("=== Generated Hypothetical Document ===")
print(hypo_doc[:600] + "...\n")

# 2. Retrieve real documents using the native HyDE retriever
retrieved_docs = hyde_retriever.invoke({"query": query})

print(f"=== Retrieved {len(retrieved_docs)} Document(s) via HyDE ===")
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Doc {i} | Source: {doc.metadata.get('source')} | Topic: {doc.metadata.get('topic')} ---")
    print(doc.page_content)
    print()

=== Generated Hypothetical Document ===
**Transformer Models and the Role of Attention in Sequence Processing**

Transformer architectures have become the dominant framework for modeling sequential data in natural language processing, computer vision, and other domains. Their core innovation is the attention mechanism, which enables the model to weigh the relevance of different elements in a sequence when computing representations for each position. This document outlines how attention is employed within transformers to process sequences, covering the mathematical formulation, architectural components, and practical implications.

-...

=== Retrieved 3 Document(s) via HyDE ===
--- Doc 1 | Source: AI | Topic: Transformers ---
The Transformer architecture, introduced in 'Attention Is All You Need' (Vaswani et al., 2017), replaces recurrent layers entirely with self-attention mechanisms. Each token attends to every other token in the sequence simultaneously, computing weighted sums based 

## 7. Context Augmentation & Generation Phase

Now we pass the retrieved real documents along with the original question to `generation_llm` to produce the final answer.

In [7]:
# Fuse retrieved document contents into context
context = "\n\n".join(doc.page_content for doc in retrieved_docs)

generation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. Answer the user's question using only the provided context. "
        "Be concise and accurate. If the context does not contain enough information, say so."
    ),
    ("human", "Context:\n{context}\n\nQuestion: {query}")
])

generation_chain = generation_prompt | generation_llm | StrOutputParser()

response = generation_chain.invoke({"context": context, "query": query})

print("=== Final Answer ===")
print(response)

=== Final Answer ===
Transformer models process sequences with self‑attention. For each token, the model creates a query vector and compares it to key vectors of all tokens in the sequence, computing dot‑product scores that are turned into attention weights. These weights are used to take weighted sums of the tokens’ value vectors, so every token can incorporate information from every other token in a single parallel step. Multi‑head attention runs several such attention operations in parallel, each learning a different representation subspace (e.g., syntactic or semantic), and then concatenates and projects the results to form the final token representation. This replaces recurrent processing and lets the model capture long‑range dependencies efficiently.


## 8. Complete End-to-End One-Line HyDE Chain

In LangChain LCEL, the whole HyDE pipeline can be executed as a single unified chain from query to final answer:

In [8]:
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)

# Full LCEL HyDE Pipeline
full_rag_hyde_chain = (
    {
        "context": hyde_retriever | format_docs,
        "query": lambda x: x["query"]
    }
    | generation_prompt
    | generation_llm
    | StrOutputParser()
)

# Invoke the complete chain
final_result = full_rag_hyde_chain.invoke({
    "query": "How do transformer models use attention to process sequences?"
})

print("=== End-to-End Pipeline Output ===")
print(final_result)

=== End-to-End Pipeline Output ===
Transformer models process sequences with **self‑attention**. For every token in the input, the model creates a query vector and compares it with key vectors of all tokens in the same sequence via dot‑product similarity. The resulting scores are normalized (softmax) to produce attention weights, which are used to compute a weighted sum of the corresponding value vectors. This gives each token a context‑aware representation that incorporates information from every other token simultaneously, enabling parallel processing of the whole sequence.

To capture different types of relationships, transformers employ **multi‑head attention**: several independent attention heads each perform the query‑key‑value computation in separate subspaces, learning distinct patterns (e.g., syntactic vs. semantic). The outputs of all heads are concatenated and linearly projected to form the final token representation, which is then passed through the rest of the model. This 